# RotatE on FB15k-237 with custom two-stage negative sampling

This notebook reproduces the **custom RotatE** pipeline used in our D3 / DL-final
submissions. It trains RotatE on FB15k-237 with three negative-sampling strategies
(`random`, `hard`, `mixed`) drawn from a scored candidate pool, then evaluates each
checkpoint globally, per slice (relation frequency, head/tail degree), and qualitatively.

## Unified configuration shared with the baselines
* Loss: `NSSALoss(margin=9.0, adversarial_temperature=1.0)` (RotatE paper default)
* Sampler: Bernoulli with `filtered=True`, `num_negs_per_pos=8`
* Model: `embedding_dim=128`, Adam `lr=1e-3`, `batch_size=1024`
* Training: 50 epochs max, early stopping on filtered val MRR, `patience=10`
* Seed: 42

Only the **negative-selection strategy** changes between runs. The companion CLI is
`code/train_rotate_custom.py`; this notebook drives it with the exact same arguments
we used to produce our reported numbers.

## How to use
1. Set Colab runtime to **GPU (T4)** (Runtime ▸ Change runtime type).
2. Run all cells in order. Section A (setup) is a one-time install.
3. Section C **smoke-trains** one run (1 epoch, ~2 min) to validate the pipeline.
4. Section D shows the **full 50-epoch commands** for the five custom runs.
5. Section E loads any available checkpoints and produces the sliced / qualitative analyses.


## A. Setup (clone repo + install dependencies)

If the cell below fails because the repo is private, upload the contents of `code/` manually
to `/content/dl_kg_project/code/` and skip the `git clone` step.

In [ ]:
import os, subprocess

REPO_URL  = "https://github.com/thaalia/dl_kg_project.git"  # private repo: clone may fail
REPO_PATH = "/content/dl_kg_project"
CODE_DIR  = os.path.join(REPO_PATH, "code")

if not os.path.isdir(CODE_DIR):
    if not os.path.isdir(REPO_PATH):
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_PATH],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            os.makedirs(CODE_DIR, exist_ok=True)
            print("git clone failed (private repo or no network access).")
            print("Manual upload required:")
            print(f"  1. In the Colab Files panel, navigate to {REPO_PATH}/code/")
            print("  2. Upload all .py files from your local code/ folder.")
            print("  3. Also upload requirements.txt at the repo root if not present.")
            print("\nClone stderr was:\n", result.stderr.strip())

os.chdir(REPO_PATH)
print("\nWorking directory:", os.getcwd())
print("Code files:", sorted(os.listdir("code")) if os.path.isdir("code") else "code/ not found yet")

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch
print("PyTorch :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Full training will be impractically slow on CPU.")

## B. Configuration

These parameters mirror the CLI flags of `code/train_rotate_custom.py` so that the notebook
produces the same artefacts as the script we report on.

In [ ]:
MODEL                   = "RotatE"
DIM                     = 128
BATCH_SIZE              = 1024
LR                      = 1e-3
EPOCHS                  = 50          # 1 for smoke; 50 for the reported runs
PATIENCE                = 10
SEED                    = 42
POOL_SIZE               = 64
NUM_NEGS                = 8
MARGIN                  = 9.0
ADVERSARIAL_TEMPERATURE = 1.0
STRATEGY                = "mixed"     # one of: random | hard | mixed
HARD_FRACTION           = 0.5         # only used when STRATEGY == 'mixed'

# Artefacts will be written to: artifacts/custom/RotatE_<strategy>(_<hard%>_<rand%>)/
print("Strategy:", STRATEGY, "| hard_fraction:", HARD_FRACTION if STRATEGY == "mixed" else "n/a")

## C. Smoke training (1 epoch, ~2 min) — validates the full pipeline

The smoke cell limits batches and val triples so it returns in a couple of minutes. Once it
completes you'll see `artifacts/custom/RotatE_mixed_50_50/summary.txt` updated with a tiny
`test_mrr` (the model has barely started learning); the goal of this step is purely to
confirm that candidate generation, scoring, selection and the loss back-prop chain run end-to-end
without errors.

In [ ]:
!python code/train_rotate_custom.py \
    --strategy {STRATEGY} \
    --hard-fraction {HARD_FRACTION} \
    --num-negs {NUM_NEGS} \
    --pool-size {POOL_SIZE} \
    --epochs 1 \
    --batch-size 256 \
    --margin {MARGIN} \
    --adversarial-temperature {ADVERSARIAL_TEMPERATURE} \
    --seed {SEED} \
    --limit-batches 10 \
    --limit-val-eval 200

## D. Full reported runs (5 strategies × 50 epochs ≈ 4–5 h total on T4)

These are the exact commands that produced the numbers in our report. Each one writes its
own folder under `artifacts/custom/`. Do **not** mix git pulls between two runs that you
want to compare directly.

In [ ]:
# Uncomment one (or more) of these to launch the full 50-epoch runs.
# Each command saves a summary, history JSON, learning-curve PNG, and trained_model.pkl.

# !python code/train_rotate_custom.py --strategy random                       --num-negs 8 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy hard                         --num-negs 8 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.3    --num-negs 8 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.5    --num-negs 8 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.7    --num-negs 8 --epochs 50 --patience 10

## E. Sliced evaluation on every available checkpoint

`evaluate_slices.py` finds every `trained_model.pkl` under `artifacts/` and produces a
`slices.json` next to it. The buckets (low / mid / high) are computed once from the training
split only and cached in `artifacts/slice_buckets.json`, ensuring the same buckets are
reused across every run.

In [ ]:
!python code/evaluate_slices.py --all

In [ ]:
# Consolidate the per-run slices.json files into a single table.
import json
from pathlib import Path
import pandas as pd

rows = []
for slice_path in sorted(Path("artifacts").rglob("slices.json")):
    data = json.load(slice_path.open())
    g = data["global"]
    row = {
        "run": str(slice_path.parent.relative_to("artifacts")),
        "mrr": g["mrr"],
        "h@1": g["hits_at_1"],
        "h@3": g["hits_at_3"],
        "h@10": g["hits_at_10"],
        "mrr_head": g["mrr_head"],
        "mrr_tail": g["mrr_tail"],
    }
    for axis in ("relation_frequency", "head_degree", "tail_degree"):
        for bucket in ("low", "mid", "high"):
            row[f"{axis}.{bucket}"] = data[axis][bucket].get("mrr")
    rows.append(row)

df = pd.DataFrame(rows).set_index("run")
df.round(4)

## F. Qualitative comparison on a handful of test triples

`qualitative_examples.py` samples a few test triples (seed=42 by default) and, for each
model, prints the filtered rank of the gold entity plus the top-K predicted candidates on
both sides. The full report is saved to `artifacts/qualitative.md`.

In [ ]:
!python code/qualitative_examples.py --num-triples 5

In [ ]:
# Preview the first ~120 lines of the qualitative report inline.
from IPython.display import Markdown
report = Path("artifacts/qualitative.md").read_text().splitlines()
Markdown("\n".join(report[:120]))

## G. Headline visualisations

Two figures we use in the report: (1) global MRR per strategy, and (2) MRR vs `tail in-degree`
bucket, which is the slice that benefits most from hard-negative sampling.

In [ ]:
import matplotlib.pyplot as plt

if df.empty:
    print("No checkpoints found; run sections C or D first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    df["mrr"].sort_values().plot.barh(ax=axes[0], color="steelblue")
    axes[0].set_title("Filtered MRR per run")
    axes[0].set_xlabel("MRR")
    axes[0].axvline(df.loc[df.index.str.contains("pykeen_RotatE")]["mrr"].mean(),
                    color="red", linestyle="--", linewidth=1, label="RotatE baseline")
    axes[0].legend()

    cols = ["tail_degree.low", "tail_degree.mid", "tail_degree.high"]
    df[cols].T.plot(ax=axes[1], marker="o")
    axes[1].set_title("MRR by tail in-degree bucket")
    axes[1].set_xticks(range(3))
    axes[1].set_xticklabels(["low", "mid", "high"])
    axes[1].set_ylabel("Filtered MRR")
    axes[1].legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    fig.savefig("artifacts/d3_headline_figures.png", dpi=150)
    plt.show()

## H. Notes for graders

* The negative-sampling logic lives in three small modules:
  * `code/negative_sampling.py` — Bernoulli corruption + train-triple filter.
  * `code/score_candidates.py` — batched scoring of candidate triples.
  * `code/select_candidates.py` — `random` / `hard` / `mixed` selection from the scored pool.
* The training driver `code/train_rotate_custom.py` calls `model.post_parameter_update()`
  after every optimiser step to keep RotatE's relation embeddings on the complex unit circle,
  matching what PyKEEN's `pipeline()` does internally. This step was essential for the
  NSSALoss to converge.
* `code/evaluate_slices.py` and `code/qualitative_examples.py` are pure post-hoc tools that
  consume a `trained_model.pkl` and do not retrain anything.
